In [56]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import string
import os
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

## 3 Sequence-to-Sequence Modeling

### Data Preprocessing

In [57]:
data = pd.read_csv('topical_chat_pairs.csv', sep='\t')
data.head()

,id,conversation_id,message,answer
0,0,1,Are you a fan of Google or Microsoft?,Both are excellent technology they are helpfu...
1,1,1,Both are excellent technology they are helpfu...,"I'm not a huge fan of Google, but I use it a..."
2,2,1,"I'm not a huge fan of Google, but I use it a...",Google provides online related services and p...
3,3,1,Google provides online related services and p...,"Yeah, their services are good. I'm just not a..."
4,4,1,"Yeah, their services are good. I'm just not a...",Google is leading the alphabet subsidiary and...


In [58]:
word2index = {}
index2word = {}

word2index['<SOS>'] = 0
word2index['<EOS>'] = 1
word2index['<PAD>'] = 2
index2word[0] = '<SOS>'
index2word[1] = '<EOS>'
index2word[2] = '<PAD>'

In [59]:
lenghts = []
unique_words = set()
for text in data['message']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
for text in data['answer']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
print(f'Number of unique words: {len(unique_words)}')
print(f'{sorted(list(unique_words))[:5]}')
AVERAGE_LENGTH = round(np.mean(lenghts))
print(f'Average length of sentences: {AVERAGE_LENGTH}')

Number of unique words: 41112
['!', '#', '$', '%', '&']
Average length of sentences: 23


In [60]:
for word in unique_words:
    index = len(word2index)
    word2index[word] = index
    index2word[index] = word
print(f'Number of words in vocabulary: {len(word2index)}')

Number of words in vocabulary: 41115


In [61]:
def collate_fn(batch):
    messages, answers = zip(*batch)
    preprocessed_messages = []
    for message in messages:
        message = word_tokenize(message.strip().lower())
        preprocessed_message = []
        preprocessed_message.append(word2index['<SOS>'])
        if len(message) <= AVERAGE_LENGTH:
            for word in message:
                preprocessed_message.append(word2index[word])
            preprocessed_message += [word2index['<PAD>']] * (AVERAGE_LENGTH - len(message))
        else:
            for word in message[:AVERAGE_LENGTH]:
                preprocessed_message.append(word2index[word])
        preprocessed_message.append(word2index['<EOS>'])
        preprocessed_messages.append(preprocessed_message)

    max_length = max(len(text) for text in answers)
    preprocessed_answers = []
    for answer in answers:
        answer = word_tokenize(answer.strip().lower())
        preprocessed_answer = []
        preprocessed_answer.append(word2index['<SOS>'])
        if len(answer) <= AVERAGE_LENGTH:
            for word in answer:
                preprocessed_answer.append(word2index[word])
            preprocessed_answer += [word2index['<PAD>']] * (AVERAGE_LENGTH - len(answer))
        else:
            for word in answer[:AVERAGE_LENGTH]:
                preprocessed_answer.append(word2index[word])
        preprocessed_answer.append(word2index['<EOS>'])
        preprocessed_answers.append(preprocessed_answer)

    preprocessed_messages = torch.tensor(preprocessed_messages, dtype=torch.long)
    preprocessed_answers = torch.tensor(preprocessed_answers, dtype=torch.long)
    return preprocessed_messages, preprocessed_answers

In [62]:
class SequenceDataset(Dataset):
    def __init__(self, messages, answers):
        if len(messages) != len(answers):
            raise ValueError("Messages and answers must have the same length.")
        if not isinstance(messages, list) or not isinstance(answers, list):
            try:
                messages = messages.tolist()
                answers = answers.tolist()
            except AttributeError:
                raise TypeError("Messages and answers must be convertible to lists.")
        self.messages = messages
        self.answers = answers

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        return self.messages[idx], self.answers[idx]

In [63]:
dataset = SequenceDataset(data['message'], data['answer'])

In [64]:
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [65]:
for messages, answers in train_loader:
    print(f'Messages batch shape: {messages.shape}')
    print(f'Answers batch shape: {answers.shape}')
    for i in range(1):
        print(f'Message {i}: {messages[i]}')
        print(f'Answer {i}: {answers[i]}')
    break

Messages batch shape: torch.Size([32, 25])
Answers batch shape: torch.Size([32, 25])
Message 0: tensor([    0, 38816, 21083, 29239, 16636, 33120,  1639, 16249, 12406, 28484,
        39282, 22564, 40680, 40783, 20502,  3106, 12406,  5109, 18503, 16249,
        17940, 16925,   501,  7084,     1])
Answer 0: tensor([    0, 31124, 15030,  3585,  6342,  7084,  6807, 20502, 34540, 38927,
        41109, 40783, 39316, 32745, 23753, 38927, 16636,  6700, 16249,  7084,
        40783, 20502, 12269, 18012,     1])
